### PromptTemplate 
* [PromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.prompt.PromptTemplate.html#langchain_core.prompts.prompt.PromptTemplate)
* [ChatPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatPromptTemplate.html#langchain_core.prompts.chat.ChatPromptTemplate)
* [ChatMessagePromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.chat.ChatMessagePromptTemplate.html#langchain_core.prompts.chat.ChatMessagePromptTemplate)
* [FewShotPromptTemplate](https://python.langchain.com/api_reference/core/prompts/langchain_core.prompts.few_shot.FewShotPromptTemplate.html#langchain_core.prompts.few_shot.FewShotPromptTemplate)
* PartialPrompt

In [ ]:
# uv add python-dotenv langchain langchain-openai

In [1]:
from dotenv import load_dotenv
import os
# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 사용할 제공자의 키만 있으면 됨 — 설정 여부와 앞 2글자만 확인용으로 출력 (키 전체는 출력하지 않음)
for name, key in (("GROQ_API_KEY", GROQ_API_KEY), ("OPENAI_API_KEY", OPENAI_API_KEY)):
    status = f"설정됨 ({key[:2]}...)" if key else "없음"
    print(f"{name}: {status}")

GROQ_API_KEY: 설정됨 (gs...)
OPENAI_API_KEY: 설정됨 (sk...)


#### LLM 설정 (Groq / OpenAI 선택)

- **Groq**와 **OpenAI** 두 가지 모델을 모두 사용할 수 있습니다. (Groq는 OpenAI 호환 API라서 `ChatOpenAI`로 연결)
- `get_llm(temperature=0.7, provider="groq")`처럼 **`provider`를 지정해 필요할 때 LLM을 다시 생성**할 수 있습니다. (`"groq"` 또는 `"openai"`)
- `provider`를 생략하면 아래 설정 셀의 `PROVIDER` 값이 기본값으로 사용됩니다.
- 이 노트북의 LLM 생성 셀은 `provider="groq"`로 지정되어 있습니다. OpenAI를 쓰려면 해당 셀의 값을 `"openai"`로 바꾸세요. (OpenAI 사용 시 요금 발생)
- 사용할 제공자의 키(`GROQ_API_KEY` 또는 `OPENAI_API_KEY`)가 `.env`에 있어야 합니다.

In [12]:
from langchain_openai import ChatOpenAI

# 기본 LLM 제공자: "groq" 또는 "openai" — get_llm() 호출 시 provider 를 생략하면 이 값을 사용
PROVIDER = "groq"

# 제공자별 설정 (Groq는 OpenAI 호환 API라서 base_url만 다르고, OpenAI는 base_url 생략 시 기본 엔드포인트 사용)
LLM_CONFIGS = {
    "groq": {
        "api_key": GROQ_API_KEY,
        "base_url": "https://api.groq.com/openai/v1",  # Groq API 엔드포인트
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": OPENAI_API_KEY,
        "base_url": None,
        "model": "gpt-5.6-luna",  # 대안: "gpt-4o"
    },
}
MAX_RETRIES = 5  # 503 등 일시 장애 자동 재시도 횟수


def get_llm(temperature: float = 0.5, seed: int | None = None, provider: str | None = None, verbose: bool = True) -> ChatOpenAI:
    """선택한 제공자(기본값: PROVIDER)의 ChatOpenAI 인스턴스를 생성합니다. verbose=True 이면 생성된 모델 정보를 출력합니다."""
    provider = provider or PROVIDER
    if provider not in LLM_CONFIGS:
        raise ValueError(f"지원하지 않는 제공자: {provider} (사용 가능: {list(LLM_CONFIGS)})")
    config = LLM_CONFIGS[provider]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {provider.upper()}_API_KEY 가 없습니다.")
    llm = ChatOpenAI(
        **config,
        temperature=temperature,
        seed=seed,  # 값을 고정하면 재현성이 향상됨 (완전히 동일한 결과는 보장되지 않음)
        max_retries=MAX_RETRIES,
    )
    if verbose:
        # 생성된 모델 정보 출력 (API 키는 출력하지 않음)
        print(f"[LLM 생성] provider={provider} | model={llm.model_name}")
    return llm

##### 1) PromptTemplate 의 from_template() 함수 사용
* 주로 LLM(텍스트 완성형 모델, ex. Ollama, GPT-3.5)과 함께 사용
* 하나의 문자열 프롬프트를 생성

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pprint import pprint

template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

llm = get_llm(temperature=0.7, provider="groq")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)

[LLM 생성] provider=groq | model=openai/gpt-oss-120b


In [4]:

chain = prompt_template | llm | StrOutputParser()
response = chain.invoke({"model_name":"ChatGPT", "count":3})
pprint(response)

('ChatGPT는 대규모 텍스트 데이터에 대해 **자기지도 학습**을 수행하여, 주어진 문맥에 맞는 다음 단어를 예측하도록 모델을 '
 '훈련합니다.  \n'
 '훈련 과정에서 **트랜스포머 아키텍처**를 사용해 단어 간의 관계와 의미를 파악하고, **다중 헤드 어텐션**을 통해 문맥 정보를 '
 '효율적으로 통합합니다.  \n'
 '이후 **지도 학습**과 **강화학습(예: RLHF)**을 통해 인간 피드백을 반영하여 답변의 정확성·일관성·안전성을 향상시킵니다.')


##### 2) PromptTemplate 결합하기
* 동일한 Prompt 패턴을 사용하지만 여러 개의 질문을 작성해서 LLM을 실행할 수도 있습니다.

In [13]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

# 문자열 템플릿 결합 (PromptTemplate + PromptTemplate + 문자열)
combined_prompt = (
              prompt_template
              + PromptTemplate.from_template("\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요")
              + "\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요."
)
combined_prompt.format(model_name="ChatGPT", count=3, language="한국어")
print(combined_prompt)

chain = combined_prompt | llm | StrOutputParser()
response = chain.invoke({"model_name":"Gemini", "count":3, "language":"한국어"})

pprint(response)

input_variables=['count', 'language', 'model_name'] input_types={} partial_variables={} template='{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요.\n\n 그리고 {model_name} 모델의 장점을 요약 정리해 주세요\n\n {model_name} 모델과 비슷한 AI 모델은 어떤 것이 있나요? 모델명은 {language}로 답변해 주세요.'
('Gemini 모델은 대량의 데이터를 기반으로 한 딥러닝 알고리즘을 사용하여 자연어 처리 및 이미지 인식 등의 다양한 작업을 수행합니다. '
 '이 모델은 여러 층의 신경망을 통해 복잡한 패턴을 학습하고, 이를 통해 사람과 유사한 방식으로 정보와 지식을 이해합니다. 또한, 지속적인 '
 '학습을 통해 성능을 개선하고 적응할 수 있는 능력을 갖추고 있습니다.\n'
 '\n'
 '**Gemini 모델의 장점:**\n'
 '1. 다양한 데이터 유형을 처리할 수 있는 멀티모달 능력.\n'
 '2. 높은 정확도와 빠른 응답 속도로 사용자의 요구를 충족.\n'
 '3. 지속적인 학습을 통해 시간이 지남에 따라 성능 향상 가능.\n'
 '\n'
 '**Gemini 모델과 비슷한 AI 모델:**\n'
 '1. GPT-4\n'
 '2. BERT\n'
 '3. CLIP\n'
 '4. T5')


#### PromptTemplate 의 파라미터를 배열 형태로 하여 여러개 사용하는 경우

In [6]:
# List Comprehension 예제
# before (적용하지 않은 경우)
result_list = []
for val in range(10):
    # 2의
    if val % 2 == 0:
        result_list.append(val ** 2)

result_list        

[0, 4, 16, 36, 64]

In [7]:
# after (적용한 경우)
result_list2 = [val**2 for val in range(10) if val % 2 == 0]
result_list2

[0, 4, 16, 36, 64]

In [8]:
template_text = "{model_name} 모델의 학습 원리를 {count} 문장으로 한국어로 답변해 주세요."

# PromptTemplate 인스턴스를 생성
prompt_template = PromptTemplate.from_template(template_text)

questions = [
    {"model_name": "GPT-4", "count": 3}, #dict
    {"model_name": "Gemini", "count": 4},
    {"model_name": "Claude", "count": 4},
]

# 여러 개의 프롬프트를 미리 생성
# q변수가 dict 타입
formatted_prompts = [prompt_template.format(**q) for q in questions]
print(formatted_prompts)  # 미리 생성된 질문 목록 확인

for prompt in formatted_prompts:
    print(type(prompt), prompt)
    response = llm.invoke(prompt)
    pprint(response.content)
    print("-" * 50)

['GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.', 'Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.', 'Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.']
<class 'str'> GPT-4 모델의 학습 원리를 3 문장으로 한국어로 답변해 주세요.
('GPT‑4는 대규모 텍스트 데이터를 사용해 **자기지도 학습**(self‑supervised learning) 방식으로 사전 학습됩니다. '
 '모델은 입력 문장의 일부를 가리고, 그 빈칸을 주변 문맥을 통해 예측하도록 훈련되어 언어의 통계적 패턴과 의미 관계를 학습합니다. '
 '이후에는 인간이 만든 **보상 모델**(reward model)과 강화학습(RLHF)을 이용해 원하는 답변 스타일과 안전성을 갖도록 '
 '미세조정합니다.')
--------------------------------------------------
<class 'str'> Gemini 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Gemini 모델은 대규모 텍스트와 이미지 데이터를 수집해 토큰화한 뒤, Transformer 기반의 신경망에 입력합니다.  \n'
 '그 후, 자기지도 학습(self‑supervised learning) 방식으로 마스크된 토큰을 예측하도록 사전 '
 '학습(pre‑training)을 진행합니다.  \n'
 '다양한 멀티모달 과제에 대해 인간 피드백을 활용한 강화학습(RLHF)으로 모델의 응답 품질과 안전성을 향상시킵니다.  \n'
 '마지막으로, 실제 서비스 환경에 맞게 도메인‑특화 데이터와 지속적인 업데이트를 통해 모델을 미세조정(fine‑tuning)합니다.')
--------------------------------------------------
<class 'str'> Claude 모델의 학습 원리를 4 문장으로 한국어로 답변해 주세요.
('Claude 모델은 대규모 텍스트 데이터를 수집하여 **사전 학습(pre

##### 2) ChatPromptTemplate
* Tuple 형태의 system, user, assistant 메시지 지원
* 여러 개의 메시지를 조합하여 LLM에게 전달 가능
* 간결성과 가독성이 높고 단순한 구조

In [14]:
# 2-튜플 형태의 메시지 목록으로 프롬프트 생성 (type, content)

from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    # role, message
    ("system", "This system is an expert in answering questions about {topic}. \
     Please provide clear and detailed explanations."),
    ("user", "{model_name} 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?"),
])

messages = chat_prompt.format_messages(topic="AI", model_name="ExaOne")
print(messages)

# 생성한 메시지를 바로 주입하여 호출하기
llm = get_llm(temperature=0.7, provider="openai") 
response = llm.invoke(messages)

print(type(response))
print(response.content)

[SystemMessage(content='This system is an expert in answering questions about AI.      Please provide clear and detailed explanations.', additional_kwargs={}, response_metadata={}), HumanMessage(content='ExaOne 모델의 학습 원리를 설명해 주세요.이 AI 모델은 어느 나라에서 개발되었나요?', additional_kwargs={}, response_metadata={})]
[LLM 생성] provider=openai | model=gpt-5.6-luna
<class 'langchain_core.messages.ai.AIMessage'>
**EXAONE(엑사원)**은 LG의 생성형 AI 모델군으로, 주로 **LG AI연구원(LG AI Research)**이 개발했습니다. 따라서 개발 국가는 **대한민국(한국)**입니다.

## EXAONE의 학습 원리

EXAONE은 버전에 따라 텍스트 전용, 멀티모달, 추론 특화 등 구조와 학습 방식이 조금씩 다르지만, 기본 원리는 다음과 같습니다.

### 1. 대규모 데이터로 사전 학습

먼저 책, 웹 문서, 논문, 코드, 뉴스 등 정제된 대규모 데이터를 사용해 언어의 통계적 패턴을 학습합니다.

가장 기본적인 학습 과제는 다음과 같습니다.

> 앞의 문맥을 보고 다음에 올 토큰(token)을 예측하기

예를 들어,

> “대한민국의 수도는 ___이다.”

라는 문장이 주어지면 모델은 문맥을 바탕으로 “서울”과 같은 다음 토큰의 확률을 계산합니다.

이 과정을 매우 많은 문장에 반복하면서 다음을 학습합니다.

- 단어와 문장의 의미
- 문법과 문체
- 문서의 구조
- 질문과 답변의 관계
- 한국어와 영어 등 여러 언어의 표현 방식
- 코드와 수학적 표현의 패턴

### 2. 트랜스포머와 어텐션 사용

EXAONE과 같은 현대적인 대규모 언어 모델은 일반적으로 *

In [ ]:
# 체인을 생성하여 호출하기
chain = chat_prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"AI", "model_name":"deepseek"})
print(type(response))
print(response)

#### 3) ChatPromptTemplate
* SystemMessagePromptTemplate와 HumanMessagePromptTemplate 클래스 사용
* 객체 지향적 접근 - Message 객체를 독립적으로 생성 가능
* 여러 조건에 따라 다른 시스템 메시지 선택

```python
if user_is_beginner:
    system_message = SystemMessagePromptTemplate.from_template("초보자를 위한 설명: {topic}")
else:
    system_message = SystemMessagePromptTemplate.from_template("전문가를 위한 상세 분석: {topic}")
```

In [15]:
# ChatMessagePromptTemplate 활용

from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    AIMessagePromptTemplate,
    ChatMessagePromptTemplate
)
from langchain_openai import ChatOpenAI

# 개별 메시지 템플릿 정의
system_message = SystemMessagePromptTemplate.from_template(
    "You are an AI expert in {topic}. Please provide clear and detailed explanations."
)
user_message = HumanMessagePromptTemplate.from_template(
    "{question}"
)
ai_message = AIMessagePromptTemplate.from_template(
    "This is an example answer about {topic}."
)

# ChatPromptTemplate로 메시지들을 묶기
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    user_message,
    ai_message
])

# 메시지 생성
messages = chat_prompt.format_messages(topic="건강", question="독감예방 접종을 꼭 맞아야 하나요?")

# LLM 호출 (공통 설정 셀의 llm 사용)
response = llm.invoke(messages)

# 결과 출력
print(response.content)

독감(인플루엔자) 예방접종은 **모든 사람이 법적으로 꼭 맞아야 하는 것은 아니지만, 대부분에게 권장**됩니다. 특히 독감으로 인한 합병증 위험이 큰 사람은 맞는 것이 좋습니다.

### 특히 접종을 권하는 경우
- **65세 이상**
- **생후 6개월 이상 어린이**, 특히 어린이집·학교에 다니는 아이
- **임신부**
- 천식·만성폐질환, 심장질환, 당뇨병, 신장·간질환, 면역저하 등 만성질환이 있는 사람
- 요양시설 거주자
- 영유아·고령자·만성질환자를 돌보는 가족이나 의료진

독감은 단순 감기보다 폐렴, 입원, 기존 질환 악화로 이어질 수 있습니다. 예방접종은 감염 자체를 100% 막지는 못하지만, **감염과 중증·입원 위험을 낮추는 데 도움이 됩니다.** 접종을 해도 걸릴 수 있지만 증상이 비교적 가볍게 지나갈 가능성이 있습니다.

### 접종 시기
- 유행 전인 **가을철**에 맞는 것이 가장 좋습니다.
- 유행이 시작된 뒤라도 아직 접종하지 않았다면 맞을 가치가 있습니다.
- 독감 백신의 효과는 시간이 지나며 감소하고 유행 바이러스가 바뀔 수 있어 **매년 접종**이 필요합니다.

### 안전성
- 접종 후 주사 부위 통증, 피로감, 미열 등이 1~2일 생길 수 있습니다.
- **주사 백신이 독감을 일으키지는 않습니다.**
- 과거 독감 백신이나 성분에 심한 알레르기 반응이 있었거나, 접종 후 길랭-바레 증후군이 발생한 적이 있다면 먼저 의사와 상담해야 합니다.
- 고열 등으로 심하게 아픈 경우에는 회복 후 접종을 고려합니다.

**결론적으로**, 건강한 성인도 접종을 권장하지만, 특히 고령자·어린이·임신부·만성질환자는 가능한 한 접종하는 것이 좋습니다. 생후 6개월 미만 아기는 접종할 수 없으므로 가족과 보호자가 예방접종을 하는 것이 중요합니다.


#### ChatMessagePromptTemplate는 여러 종류의 메시지(시스템, 인간, AI)를 조합하여 복잡한 프롬프트를 생성할 때 유용합니다.
* SystemMessagePromptTemplate: 이 템플릿은 AI 모델에게 역할을 부여하거나 전반적인 규칙을 설정하는 시스템 메시지를 만듭니다. 위의 예시에서는 "번역을 도와주는 유용한 도우미"라는 역할을 지정합니다.
* HumanMessagePromptTemplate: 이 템플릿은 사용자의 질문이나 요청을 담는 인간 메시지를 만듭니다. 아래의 예시에서는 번역할 텍스트를 입력받습니다.
* ChatPromptTemplate.from_messages: 이 클래스 메서드는 시스템 메시지, 인간 메시지 등 여러 종류의 MessagePromptTemplate 객체들을 리스트로 받아 하나의 채팅 프롬프트 템플릿으로 통합합니다.
* format_messages: 이 메서드는 정의된 템플릿에 실제 값을 채워 넣어 [SystemMessage, HumanMessage] 형태의 리스트를 반환합니다. 이 리스트는 채팅 모델(Chat Model) 에 바로 전달될 수 있습니다.

In [ ]:
# 필요한 라이브러리 임포트
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

# 1. SystemMessagePromptTemplate와 HumanMessagePromptTemplate 생성
# SystemMessagePromptTemplate는 모델의 페르소나 또는 기본 지침을 설정합니다.
system_template = "You are a helpful assistant that translates {input_language} to {output_language}."
system_message_prompt = SystemMessagePromptTemplate.from_template(system_template)

# HumanMessagePromptTemplate는 사용자로부터 받는 입력 프롬프트를 정의합니다.
human_template = "{text_to_translate}"
human_message_prompt = HumanMessagePromptTemplate.from_template(human_template)

# 2. ChatPromptTemplate 생성
# 위에서 만든 두 템플릿을 리스트로 묶어 ChatPromptTemplate을 만듭니다.
chat_prompt_template = ChatPromptTemplate.from_messages([system_message_prompt, human_message_prompt])

# 3. 프롬프트 포맷팅
# chat_prompt_template.format_messages()를 사용하여 최종 메시지 리스트를 생성합니다.
# 이 함수는 딕셔너리 형태의 입력 변수를 받습니다.
formatted_prompt = chat_prompt_template.format_messages(
    input_language="English",
    output_language="Korean",
    text_to_translate="Chaining can mean making multiple LLM calls in a sequence. Language models are often non deterministic and can make errors, so making multiple calls to check previous outputs or to break down larger tasks into bite-sized steps can improve results."
)

# 4. 결과 출력
print(formatted_prompt)

# LLM 호출
response = llm.invoke(formatted_prompt)

# 결과 출력
print(response.content)

##### 4) FewShotPromptTemplate
* FewShotPromptTemplate은 모델이 특정 형식을 따르게 하거나, 일관된 응답을 생성하도록 유도할 때 유용합니다.
* 도메인 지식이 필요하거나, AI가 오답을 줄이고 더 신뢰할 만한 답변을 생성하도록 해야 할 때 효과적입니다.

##### 4-1) PromptTemplate을 사용하지 않는 경우

In [16]:
# PromptTemplate을 사용하지 않는 경우

# model
print(llm.model_name)

# chain 실행
result = llm.invoke("태양계의 행성들을 간략히 정리해 주세요.")

print(type(result))
print(result.content)

gpt-5.6-luna
<class 'langchain_core.messages.ai.AIMessage'>
태양계의 행성은 태양에서 가까운 순서로 다음과 같습니다.

| 순서 | 행성 | 특징 |
|---|---|---|
| 1 | **수성** | 태양에 가장 가까운 작은 암석 행성. 대기가 거의 없어 낮과 밤의 온도 차가 큼 |
| 2 | **금성** | 두꺼운 이산화탄소 대기로 인해 태양계에서 가장 뜨거운 행성 |
| 3 | **지구** | 액체 상태의 물과 생명체가 존재하는 것으로 알려진 행성 |
| 4 | **화성** | 표면의 산화철 때문에 붉게 보이며, 과거 물의 흔적이 있음 |
| 5 | **목성** | 가장 큰 행성. 주로 수소와 헬륨으로 이루어진 가스 행성 |
| 6 | **토성** | 아름답고 뚜렷한 얼음·먼지 고리를 가진 가스 행성 |
| 7 | **천왕성** | 자전축이 크게 기울어 옆으로 누운 듯 회전하는 얼음 거대 행성 |
| 8 | **해왕성** | 태양에서 가장 먼 행성. 매우 강한 바람이 부는 푸른 얼음 거대 행성 |

- **수성·금성·지구·화성**: 지구형 행성(암석 행성)
- **목성·토성**: 가스 거대 행성
- **천왕성·해왕성**: 얼음 거대 행성
- **명왕성**은 2006년부터 행성이 아닌 **왜행성**으로 분류됩니다.


##### 4-2) FewShotChatMessagePromptTemplate 사용하는 경우

In [ ]:
# FewShotChatMessagePromptTemplate 사용하는 경우
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

# 예시는 시스템 프롬프트(초등학생 눈높이)와 같은 톤·같은 형식(제목 + 비유 글머리표 + 한 줄 정리)으로 통일
examples_list = [
    {
        "input": "뉴턴의 운동 법칙을 요약해 주세요.",
        "output": """### 뉴턴의 운동 법칙
- **관성**: 달리던 버스가 갑자기 멈추면 몸이 앞으로 쏠리죠? 움직이던 건 계속 움직이려 해요.
- **가속도**: 같은 힘으로 밀면 가벼운 공이 무거운 공보다 더 빨리 굴러가요.
- **작용-반작용**: 벽을 밀면 벽도 나를 똑같이 밀어요.
 한 줄 정리: 힘은 항상 짝으로 움직여요!"""
    },
    {
        "input": "지구의 대기 구성 요소를 알려주세요.",
        "output": """### 지구 대기의 구성
- **질소 (78%)**: 공기의 대부분이에요. 거의 반응하지 않는 얌전한 기체예요.
- **산소 (21%)**: 우리가 숨 쉴 때 꼭 필요해요.
- **이산화탄소 (0.04%)**: 아주 조금이지만 식물이 자라는 데 필요해요.
 한 줄 정리: 공기의 거의 전부는 질소와 산소예요!"""
    }
]

# 예제 프롬프트 템플릿
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}"),
        ("ai", "{output}"),
    ]
)

# FewShotChatMessagePromptTemplate 적용
few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples_list,
)

# 최종 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 초등학생도 쉽게 이해할 수 있도록 쉽게 설명하는 과학 교육자입니다."),
        few_shot_prompt,
        ("human", "{input}"),
    ]
)

# 모델 생성 및 체인 구성
chain = final_prompt | llm
print(llm.model_name)
print(chain)

# 테스트 실행
result = chain.invoke({"input": "전기의 발생 원리에 대하여 설명해 줘"})
print(result.content)

#### 5-1) PartialPrompt 
* 프롬프트를 더 동적으로 활용할 수 있으며, AI 응답을 더 일관성 있게 조정 가능함

In [17]:
from datetime import datetime
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# 프롬프트 템플릿 정의 (부분 변수 적용)
prompt = PromptTemplate(
    template="{season}에 일어나는 대표적인 지구과학 현상은 {phenomenon}이 맞나요? \
        {season}에 주로 발생하는 지구과학 현상을 3개 알려주세요",
    input_variables=["phenomenon"],  # 사용자 입력 필요
    # 함수 자체를 전달 → format() 호출 시점마다 현재 계절을 계산
    partial_variables={"season": get_current_season}  # 동적으로 계절 값 할당
)


# 특정 계절의 현상 질의
query = prompt.format(phenomenon="태풍 발생")
llm = get_llm(temperature=0.7, provider="openai")  # 제공자를 직접 지정 (키/모델명은 공통 설정 셀의 get_llm 사용)
result = llm.invoke(query)


# 결과 출력
print(f" 프롬프트: {query}")
print(f" 모델 응답: {result.content}")

[LLM 생성] provider=openai | model=gpt-5.6-luna
 프롬프트: 가을에 일어나는 대표적인 지구과학 현상은 태풍 발생이 맞나요?         가을에 주로 발생하는 지구과학 현상을 3개 알려주세요
 모델 응답: 네. **태풍은 가을에 자주 발생하는 대표적인 지구과학 현상**이지만, 가을에만 발생하는 것은 아닙니다. 우리나라에서는 보통 **8~10월**, 특히 늦여름부터 초가을에 태풍의 영향을 많이 받습니다.

가을에 주로 나타나는 지구과학 현상 3가지는 다음과 같습니다.

1. **태풍 발생 및 접근**
   - 여름 동안 데워진 바닷물에서 강한 열에너지를 얻어 발생합니다.
   - 북서태평양에서는 주로 8~10월에 태풍 활동이 활발합니다.

2. **추분**
   - 매년 **9월 22~23일경** 태양이 적도 위를 지나면서 낮과 밤의 길이가 거의 같아집니다.
   - 추분 이후에는 북반구에서 낮이 점점 짧아지고 기온이 내려갑니다.

3. **이동성 고기압과 큰 일교차**
   - 가을에는 차고 건조한 고기압과 따뜻한 공기가 번갈아 이동합니다.
   - 맑고 건조한 날이 많아지고, 낮과 밤의 기온 차가 커지며 첫서리가 나타나기도 합니다.


In [ ]:
from datetime import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

# 계절을 결정하는 함수 (남반구/북반구 고려)
def get_current_season(hemisphere="north"):
    month = datetime.now().month

    if hemisphere == "north":  # 북반구 (기본값)
        if 3 <= month <= 5:
            return "봄"
        elif 6 <= month <= 8:
            return "여름"
        elif 9 <= month <= 11:
            return "가을"
        else:
            return "겨울"
    else:  # 남반구 (계절 반대)
        if 3 <= month <= 5:
            return "가을"
        elif 6 <= month <= 8:
            return "겨울"
        elif 9 <= month <= 11:
            return "봄"
        else:
            return "여름"

# Step 1: 현재 계절 결정
season_name = get_current_season()
#season_name = get_current_season("south")  # 계절 값 얻기
print(f"현재 계절: {season_name}")

In [ ]:

# Step 2: 해당 계절의 자연 현상 추천
prompt2 = ChatPromptTemplate.from_template(
    "{season}에 주로 발생하는 대표적인 지구과학 현상 3가지를 알려주세요. "
    "각 현상에 대해 간단한 설명을 포함해주세요."
)

print(llm.model_name)
# 체인 2: 자연 현상 추천 (입력: 계절 → 출력: 자연 현상 목록)
chain2 = (
    {"season": lambda x : season_name}  # chain2의 출력을 season 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: 현재 계절에 따른 자연 현상 추천
response = chain2.invoke({})
print(f"\n {season_name}에 발생하는 자연 현상:\n{response}")

#### 5-2) PartialPrompt 
* API 호출 데이터, 시간 정보, 사용자 정보 등을 반영할 때 매우 유용함

In [18]:
import requests
from datetime import datetime
from langchain_core.prompts import PromptTemplate

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/USD"


def get_exchange_rate() -> str:
    """USD→KRW 환율과 API 기준 일자를 문자열로 반환합니다."""
    response = requests.get(EXCHANGE_API_URL, timeout=10)
    response.raise_for_status()
    data = response.json()
    base_date = data.get("date", datetime.now().strftime("%Y-%m-%d"))
    return f"{base_date} 기준 1달러 = {data['rates']['KRW']}원"


# Partial Prompt 활용 (함수 자체를 전달 → format() 호출 시점마다 실행)
prompt = PromptTemplate(
    template=(
        "오늘 날짜: {today}\n"
        "환율 정보: {info}\n\n"
        "위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. "
        "제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요."
    ),
    input_variables=[],  # 사용자 입력 없음
    partial_variables={
        "today": lambda: datetime.now().strftime("%Y-%m-%d"),
        "info": get_exchange_rate,  # API에서 가져온 데이터 자동 반영
    },
)

# LLM 모델명
print(llm.model_name)

# 프롬프트를 한 번만 생성해 재사용 (API 중복 호출 방지)
formatted = prompt.format()

# 모델에 프롬프트 전달 및 응답 받기
response = llm.invoke(formatted)

# 결과 출력
print(" 프롬프트:", formatted)
print(" 모델 응답:", response.content)

gpt-5.6-luna
 프롬프트: 오늘 날짜: 2026-10-08
환율 정보: 2026-10-08 기준 1달러 = 1339.09원

위 데이터만 근거로 환율 상황을 간단히 분석해 주세요. 제공되지 않은 과거 수치나 뉴스는 추측해서 쓰지 마세요.
 모델 응답: 2026년 10월 8일 기준 환율은 **1달러 = 1,339.09원**입니다.

- 달러 1달러를 매수하려면 약 **1,339원**이 필요합니다.
- 100달러는 약 **133,909원**, 1,000달러는 약 **1,339,090원**입니다.  
  *(수수료와 환전 우대는 제외)*
- 과거 환율 자료가 없으므로 환율의 상승·하락 여부나 고점·저점, 추세는 판단할 수 없습니다.
